# 🧬 MAFFT Tool Example

This notebook demonstrates how to align multiple protein or nucleotide sequences using **MAFFT**.

## 📖 What is MAFFT?

[MAFFT](https://mafft.cbrc.jp/alignment/software/) (Multiple Alignment using Fast Fourier Transform) is one of the most widely used multiple sequence alignment tools in bioinformatics. It inserts gap characters to maximize similarity across sequences, revealing conserved regions, evolutionary relationships, and functional domains.

### Key Features:
- **Fast & accurate** -- Optimized algorithms for both speed and alignment quality
- **Multiple methods** -- Auto-selects the best strategy based on input size
- **Versatile** -- Works with both protein and nucleotide sequences
- **Scalable** -- Handles alignments from a few sequences to thousands

## 📥 Imports

## 📦 Shared Data Types

### `MSA`
A multiple sequence alignment result with alignment data and metadata.

| Field / Property | Type | Description |
|---|---|---|
| `aligned_sequences` | `List[str]` | The aligned sequences with gap characters (`-`) |
| `original_sequences` | `List[str]` | Original sequences without gap characters |
| `sequence_ids` | `List[str]` | Sequence identifiers |
| `alignment_length` | `int` | Number of columns in the alignment |
| `num_sequences` | `int` | Number of sequences in the alignment |
| `total_gaps` | `int` | Total number of gap characters in the alignment |
| `average_gap_fraction` | `float` | Average fraction of gaps across all sequences |

In [1]:
from bio_programming_tools.tools.sequence_alignment.mafft.mafft import run_mafft_align, MafftInput, MafftConfig

## 🔍 1. Align Hemoglobin Sequences

Align hemoglobin subunit alpha sequences from multiple species to identify conserved regions.

### API Reference

**`MafftInput`**

| Field | Type | Default | Description |
|---|---|---|---|
| `sequences` | `List[str]` | *(required)* | List of sequences to align (minimum 2 required) |
| `sequence_ids` | `Optional[List[str]]` | `None` | Optional sequence identifiers (defaults to `seq_0`, `seq_1`, ...) |

**`MafftConfig`**

| Field | Type | Default | Description |
|---|---|---|---|
| `align_method` | `Literal["auto", "localpair", "globalpair", "genafpair"]` | `"auto"` | Alignment method: `"auto"`, `"localpair"` (L-INS-i), `"globalpair"` (G-INS-i), or `"genafpair"` (E-INS-i) |
| `max_iterations` | `int` | `0` | Maximum iterative refinement cycles (`0` is default, higher values are more accurate) |

**`MafftOutput`**

| Field | Type | Description |
|---|---|---|
| `msa` | `MSA` | The multiple sequence alignment result containing aligned sequences, sequence IDs, and original unaligned sequences |

In [2]:
# Hemoglobin subunit alpha sequences from different species
sequences = [
    "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH",  # Human
    "MVLSGEDKSNIKAAWGKIGGHGAEYGAEALERMFASFPTTKTYFPHFDVSH",  # Horse
    "MVLSAADKGNVKAAWGKVGGHAAEYGAEALERMFLSFPTTKTYFPHFDLSH",  # Bovine
    "MVLSPADKTNVKAAWSKVGGHAGEYGAEALERMFLGFPTTKTYFPHFDLSH",  # Gorilla
    "MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLST",  # Human beta
]
sequence_ids = ["Human_alpha", "Horse_alpha", "Bovine_alpha", "Gorilla_alpha", "Human_beta"]

inputs = MafftInput(sequences=sequences, sequence_ids=sequence_ids)
config = MafftConfig(align_method="auto", threads=2)

result = run_mafft_align(inputs, config)

# Display alignment summary
print(f"Number of sequences: {result.msa.num_sequences}")
print(f"Alignment length:    {result.msa.alignment_length}")
print(f"Total gaps:          {result.msa.total_gaps}")
print(f"Avg gap fraction:    {result.msa.average_gap_fraction:.3f}")

# Display the aligned sequences
print(f"\nAligned sequences:")
for seq_id, seq in result.msa.iter_with_ids():
    print(f"  {seq_id:16s} {seq}")

Number of sequences: 5
Alignment length:    53
Total gaps:          10
Avg gap fraction:    0.038

Aligned sequences:
  Human_alpha      MV-LSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHF-DLSH
  Horse_alpha      MV-LSGEDKSNIKAAWGKIGGHGAEYGAEALERMFASFPTTKTYFPHF-DVSH
  Bovine_alpha     MV-LSAADKGNVKAAWGKVGGHAAEYGAEALERMFLSFPTTKTYFPHF-DLSH
  Gorilla_alpha    MV-LSPADKTNVKAAWSKVGGHAGEYGAEALERMFLGFPTTKTYFPHF-DLSH
  Human_beta       MVHLTPEEKSAVTALWGKV--NVDEVGGEALGRLLVVYPWTQRFFESFGDLST


## 🔬 2. Analyze Conservation

The MSA object provides methods to analyze alignment quality and positional conservation. Conservation scores indicate how well a position is preserved across sequences — a score of 1.0 means all sequences share the same residue at that position.

In [3]:
# Analyze conservation at each position
msa = result.msa

# Find fully conserved positions (conservation = 1.0)
conserved_positions = []
for pos in range(msa.alignment_length):
    score = msa.get_conservation(pos)
    if score == 1.0:
        conserved_positions.append(pos)

print(f"Fully conserved positions: {len(conserved_positions)} / {msa.alignment_length}")
print(f"Conservation rate: {len(conserved_positions) / msa.alignment_length:.1%}")

# Show conservation for the first 20 positions
print(f"\nPer-position conservation (first 20 columns):")
print(f"  Position:     ", "".join(f"{i % 10}" for i in range(20)))
print(f"  Conservation: ", "".join(
    "*" if msa.get_conservation(i) == 1.0 else
    "." if msa.get_conservation(i) >= 0.6 else
    " " for i in range(20)
))

# Examine residue frequencies at a specific position
pos = 0
freqs = msa.get_position_frequencies(pos)
print(f"\nResidue frequencies at position {pos}: {freqs}")
column = msa.get_column(pos)
print(f"Residues at position {pos}: {column}")

Fully conserved positions: 22 / 53
Conservation rate: 41.5%

Per-position conservation (first 20 columns):
  Position:      01234567890123456789
  Conservation:  ****....* ...*.*.*.*

Residue frequencies at position 0: {'M': 1.0}
Residues at position 0: ['M', 'M', 'M', 'M', 'M']


## 💾 3. Export Results

Save the alignment to a file for downstream analysis.

### Supported formats:
- **FASTA** -- Standard aligned FASTA format
- **A3M** -- Compact alignment format used by structure prediction tools (e.g., AlphaFold)

The exported MSA can be used for phylogenetic analysis, conservation studies, or as input for structure prediction.

In [4]:
# Export as FASTA
result.export("hemoglobin_alignment", export_path="./example_output", file_format="fasta")
print("Exported to ./example_output/hemoglobin_alignment.fasta")

# Preview the FASTA content
print(f"\nFASTA preview:")
fasta_str = result.msa.to_fasta_string()
for line in fasta_str.split("\n")[:6]:  # Show first 3 sequences
    print(f"  {line}")

Exported to ./example_output/hemoglobin_alignment.fasta

FASTA preview:
  >Human_alpha
  MV-LSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHF-DLSH
  >Horse_alpha
  MV-LSGEDKSNIKAAWGKIGGHGAEYGAEALERMFASFPTTKTYFPHF-DVSH
  >Bovine_alpha
  MV-LSAADKGNVKAAWGKVGGHAAEYGAEALERMFLSFPTTKTYFPHF-DLSH
